In [ ]:
# Cell 1 — Install
# Restart the kernel after this cell completes, then run Cell 2 onward
!pip uninstall torchao -y -q 2>/dev/null; true
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install "trl==0.15.2" "transformers==4.46.3" "peft==0.14.0" "accelerate==1.2.1" "datasets==3.5.0" requests matplotlib -q
print("Done. Restart kernel now, then run from Cell 2.")

In [ ]:
# Cell 2 — Authentication
import os

hf_token = os.environ.get("HF_TOKEN", "")

if not hf_token:
    print("WARNING: HF_TOKEN not set — add it as a Space secret named HF_TOKEN")
else:
    from huggingface_hub import login
    login(token=hf_token)
    print("HF_TOKEN loaded and logged in to HuggingFace Hub.")

# Make globally accessible for all downstream cells
import builtins
builtins.hf_token = hf_token

In [ ]:
# Cell 3 — Config
import os, builtins

SPACE_URL         = "https://Bharath-1608-social-agent-negotiation-v1.hf.space"
TRAIN_TASKS       = ["single-round-consensus", "adversarial-information", "opioid-overdose"]

MODEL_NAME        = "unsloth/Qwen2.5-0.5B-Instruct"
HF_REPO           = "Bharath-1608/negotiation-agent-grpo"
HF_TOKEN          = getattr(builtins, "hf_token", os.environ.get("HF_TOKEN", ""))

N_EPOCHS          = 3
EPISODES_PER_TASK = 10
GROUP_SIZE        = 4
MAX_TURNS         = 20
LORA_R            = 16
LORA_ALPHA        = 32

VALID_ACTIONS = [
    "share_information", "propose_consensus", "accept_consensus",
    "reject_consensus", "challenge_proposal", "request_clarification",
    "flag_bias", "flag_agenda",
]

MEDICAL_KEYWORDS = [
    "patient", "diagnosis", "treatment", "medication", "dose",
    "symptoms", "vitals", "triage", "consensus", "evidence",
    "protocol", "critical", "stable", "urgent", "contraindication",
    "allergy", "imaging", "labs", "monitor", "consult",
]

print(f"Model          : {MODEL_NAME}")
print(f"Epochs         : {N_EPOCHS}")
print(f"Episodes/task  : {EPISODES_PER_TASK}")
print(f"Max turns      : {MAX_TURNS}")
print(f"Tasks          : {TRAIN_TASKS}")
print(f"HF_TOKEN set   : {bool(HF_TOKEN)}")

In [ ]:
# Cell 4 — Model load
import os, builtins, torch
from unsloth import FastLanguageModel
from huggingface_hub import login

MODEL_NAME = "unsloth/Qwen2.5-0.5B-Instruct"
LORA_R     = 16
LORA_ALPHA = 32
HF_TOKEN   = getattr(builtins, "hf_token", os.environ.get("HF_TOKEN", ""))

if HF_TOKEN:
    login(token=HF_TOKEN)

print(f"Loading {MODEL_NAME} with 4-bit quantisation...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = 2048,
    load_in_4bit   = True,
    dtype          = None,
    token          = HF_TOKEN or None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
)

# Qwen requires left-padding for batch generation
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,}  ({100*trainable/total:.2f}%)")
print(f"Total    : {total:,}")
print("Model ready.")

In [ ]:
# Cell 5 — Rollout function
import torch, requests, json, re

SPACE_URL   = "https://Bharath-1608-social-agent-negotiation-v1.hf.space"
MAX_TURNS   = 20
VALID_ACTIONS = [
    "share_information", "propose_consensus", "accept_consensus",
    "reject_consensus", "challenge_proposal", "request_clarification",
    "flag_bias", "flag_agenda",
]

SYSTEM_PROMPT = (
    "You are a doctor AI agent negotiating a medical decision with another doctor AI.\n"
    "Output ONLY valid JSON — no markdown, no explanation, no surrounding text.\n\n"
    "STANDARD ACTIONS (share_information, propose_consensus, accept_consensus, "
    "reject_consensus, challenge_proposal, request_clarification):\n"
    '{"action_type": "<action>", "content": "<your message>", "reasoning": "<your reasoning>"}\n\n'
    "FLAG_BIAS — use when you detect misleading framing:\n"
    '{"action_type": "flag_bias", "content": "<msg>", "reasoning": "<why>", '
    '"bias_location": "<where>", "bias_direction": "<toward what>", "bias_correction": "<fix>"}\n\n'
    "FLAG_AGENDA — use when other agent has a hidden mandate:\n"
    '{"action_type": "flag_agenda", "content": "<msg>", "reasoning": "<why>", '
    '"agenda_type": "<cost_cutter or aggressive_treater>", '
    '"agenda_evidence": "<evidence>", "agenda_counter": "<counter>"}\n\n'
    "Rules:\n"
    "- content >= 50 chars, reasoning >= 30 chars\n"
    "- propose_consensus only when ready to commit\n"
    "- accept_consensus/reject_consensus only when proposal is on the table\n"
    "- NEVER include extra keys beyond those shown"
)

FALLBACK_ACTION = {
    "action_type": "share_information",
    "content": "I need to share my clinical findings. Based on my private information, I believe we must reach consensus on the best treatment for this patient.",
    "reasoning": "Sharing private information to progress the negotiation toward a correct joint decision.",
}

FLAG_FALLBACK_FIELDS = {
    "bias_location": "not specified", "bias_direction": "not specified", "bias_correction": "not specified",
    "agenda_type": "cost_cutter", "agenda_evidence": "not specified", "agenda_counter": "not specified",
}

def _parse_action(text):
    try:
        p = json.loads(text)
        if isinstance(p, dict): return p
    except Exception:
        pass
    try:
        m = re.search(r"\{[^{}]*\}", text, re.DOTALL)
        if m:
            p = json.loads(m.group())
            if isinstance(p, dict): return p
    except Exception:
        pass
    return None

def rollout_episode(model, tokenizer, task_id):
    all_prompts, all_completions, history = [], [], []
    session_id = None

    try:
        resp = requests.post(f"{SPACE_URL}/reset", json={"task_id": task_id}, timeout=30)
        if resp.status_code == 503:
            print(f"Space sleeping — wake at {SPACE_URL}"); return [], [], 0.05
        resp.raise_for_status()
        state      = resp.json()
        session_id = state.get("session_id")
    except Exception as e:
        print(f"[rollout] Reset failed {task_id}: {e}"); return [], [], 0.05

    obs_a = state.get("obs_agent_a", {})
    obs_b = state.get("obs_agent_b", {})

    for turn in range(MAX_TURNS):
        agent_id = "agent_a" if turn % 2 == 0 else "agent_b"
        obs      = obs_a if agent_id == "agent_a" else obs_b

        task_desc    = obs.get("task_description", "Medical negotiation task.")
        private_info = json.dumps(obs.get("private_information", {}), indent=2)[:600]
        available    = obs.get("available_actions", VALID_ACTIONS)
        phase        = obs.get("current_phase", "triage")
        phase_turn   = obs.get("phase_turn", 0)

        history_str = "".join(
            f"  [{m['agent_id']}] {m['action_type']}: {m['content'][:200]}\n"
            for m in history[-6:]
        ) or "  (No messages yet — you go first.)"

        prompt = (
            SYSTEM_PROMPT + "\n\n"
            f"Task: {task_desc}\n"
            f"Phase: {phase} (turn {phase_turn})\n"
            f"Your private information:\n{private_info}\n\n"
            f"Conversation so far:\n{history_str}\n"
            f"Available actions: {available}\n"
            f"Your turn as {agent_id}. Respond with valid JSON only."
        )

        try:
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1800).to(model.device)
            with torch.no_grad():
                out = model.generate(
                    **inputs, max_new_tokens=256, do_sample=True,
                    temperature=0.7, top_p=0.9, pad_token_id=tokenizer.pad_token_id,
                )
            completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        except Exception as e:
            print(f"[rollout] Generate failed turn {turn}: {e}")
            completion = json.dumps(FALLBACK_ACTION)

        all_prompts.append(prompt)
        all_completions.append(completion)

        parsed = _parse_action(completion)
        action = parsed if (parsed and parsed.get("action_type") in VALID_ACTIONS) else dict(FALLBACK_ACTION)

        if action["action_type"] not in available:
            action["action_type"] = "share_information" if "share_information" in available else (available[0] if available else "share_information")

        step_payload = {
            "session_id": session_id,
            "action": {
                "agent_id":    agent_id,
                "action_type": action["action_type"],
                "content":     str(action.get("content", FALLBACK_ACTION["content"]))[:1000],
                "reasoning":   str(action.get("reasoning", FALLBACK_ACTION["reasoning"]))[:500],
            }
        }

        if action["action_type"] == "flag_bias":
            for f in ("bias_location", "bias_direction", "bias_correction"):
                step_payload["action"][f] = str(action.get(f, FLAG_FALLBACK_FIELDS[f]))[:300]
        if action["action_type"] == "flag_agenda":
            for f in ("agenda_type", "agenda_evidence", "agenda_counter"):
                step_payload["action"][f] = str(action.get(f, FLAG_FALLBACK_FIELDS[f]))[:300]

        try:
            step_resp = requests.post(f"{SPACE_URL}/step", json=step_payload, timeout=30)
            if step_resp.status_code == 503:
                return all_prompts, all_completions, 0.05
            if step_resp.status_code == 422:
                print(f"[rollout] 422 on turn {turn} — check flag fields")
                continue
            step_resp.raise_for_status()
            step_data = step_resp.json()
        except Exception as e:
            print(f"[rollout] Step failed turn {turn}: {e}"); continue

        obs_a = step_data.get("obs_agent_a", obs_a)
        obs_b = step_data.get("obs_agent_b", obs_b)
        history.append({"agent_id": agent_id, "action_type": action["action_type"], "content": action.get("content","")})

        if step_data.get("done", False):
            ep = step_data.get("episode_result") or {}
            if ep.get("total_reward"):
                return all_prompts, all_completions, float(ep["total_reward"])
            try:
                s = requests.get(f"{SPACE_URL}/state", params={"session_id": session_id}, timeout=30).json()
                return all_prompts, all_completions, float(s.get("cumulative_reward", 0.05))
            except Exception:
                return all_prompts, all_completions, 0.05

    try:
        s = requests.get(f"{SPACE_URL}/state", params={"session_id": session_id}, timeout=30).json()
        return all_prompts, all_completions, float(s.get("cumulative_reward", 0.05))
    except Exception:
        return all_prompts, all_completions, 0.05

print("rollout_episode() defined.")
try:
    h = requests.get(f"{SPACE_URL}/health", timeout=15).json()
    print(f"Environment healthy — {h.get('tasks_available','?')} tasks, reward range {h.get('reward_range')}")
except Exception as e:
    print(f"Cannot reach environment: {e}")

In [ ]:
# Cell 6 — Reward function (deterministic, no API calls)
import json, re

VALID_ACTIONS = [
    "share_information", "propose_consensus", "accept_consensus",
    "reject_consensus", "challenge_proposal", "request_clarification",
    "flag_bias", "flag_agenda",
]
MEDICAL_KEYWORDS = [
    "patient", "diagnosis", "treatment", "medication", "dose",
    "symptoms", "vitals", "triage", "consensus", "evidence",
    "protocol", "critical", "stable", "urgent", "contraindication",
    "allergy", "imaging", "labs", "monitor", "consult",
]
ADVERSARIAL_ACTIONS = {"flag_agenda", "flag_bias", "challenge_proposal"}

def compute_rewards(prompts, completions, task_ids=None, **kwargs):
    rewards = []
    for completion in completions:
        score, parsed = 0.0, None
        try:
            parsed = json.loads(completion)
        except Exception:
            pass
        if parsed is None:
            try:
                m = re.search(r"\{[^{}]*\}", completion, re.DOTALL)
                if m: parsed = json.loads(m.group())
            except Exception:
                pass
        if isinstance(parsed, dict):
            score += 0.3
            atype   = parsed.get("action_type", "")
            content = parsed.get("content", "")
            reason  = parsed.get("reasoning", "")
            if atype in VALID_ACTIONS:        score += 0.2
            if len(str(content))  > 50:       score += 0.2
            if len(str(reason))   > 30:       score += 0.2
            if any(k in str(content).lower() for k in MEDICAL_KEYWORDS): score += 0.1
            if atype in ADVERSARIAL_ACTIONS:  score = min(1.0, score + 0.15)
        rewards.append(round(min(1.0, max(0.0, score)), 4))
    return rewards

# Smoke test
_t = [
    '{"action_type":"share_information","content":"The patient shows critical symptoms including elevated troponin and hypotension.","reasoning":"Sharing lab results to reach correct consensus."}',
    'not json',
    '{"action_type":"flag_agenda","content":"I am flagging a cost-cutting institutional bias — patient welfare must override the mandate in this critical triage decision.","reasoning":"Detecting agenda bias before consensus."}',
]
_s = compute_rewards(None, _t)
assert _s[0] == 1.0, _s[0]
assert _s[1] == 0.0, _s[1]
assert _s[2] == 1.0, _s[2]
print(f"compute_rewards() ready. Smoke test: {_s}")

In [ ]:
# Cell 7 — GRPO training (fully self-contained)
import os, builtins, json, re, torch, requests
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

# ── Constants (redefined for self-containment) ────────────────────────────────
SPACE_URL         = "https://Bharath-1608-social-agent-negotiation-v1.hf.space"
TRAIN_TASKS       = ["single-round-consensus", "adversarial-information", "opioid-overdose"]
N_EPOCHS          = 3
EPISODES_PER_TASK = 10
GROUP_SIZE        = 4
MAX_TURNS         = 20
HF_TOKEN          = getattr(builtins, "hf_token", os.environ.get("HF_TOKEN", ""))

VALID_ACTIONS = [
    "share_information", "propose_consensus", "accept_consensus",
    "reject_consensus", "challenge_proposal", "request_clarification",
    "flag_bias", "flag_agenda",
]
MEDICAL_KEYWORDS = [
    "patient", "diagnosis", "treatment", "medication", "dose",
    "symptoms", "vitals", "triage", "consensus", "evidence",
    "protocol", "critical", "stable", "urgent", "contraindication",
    "allergy", "imaging", "labs", "monitor", "consult",
]
ADVERSARIAL_ACTIONS = {"flag_agenda", "flag_bias", "challenge_proposal"}

SYSTEM_PROMPT = (
    "You are a doctor AI agent negotiating a medical decision with another doctor AI.\n"
    "Output ONLY valid JSON — no markdown, no explanation, no surrounding text.\n\n"
    "STANDARD ACTIONS (share_information, propose_consensus, accept_consensus, "
    "reject_consensus, challenge_proposal, request_clarification):\n"
    '{"action_type": "<action>", "content": "<your message>", "reasoning": "<your reasoning>"}\n\n'
    "FLAG_BIAS:\n"
    '{"action_type": "flag_bias", "content": "<msg>", "reasoning": "<why>", '
    '"bias_location": "<where>", "bias_direction": "<toward what>", "bias_correction": "<fix>"}\n\n'
    "FLAG_AGENDA:\n"
    '{"action_type": "flag_agenda", "content": "<msg>", "reasoning": "<why>", '
    '"agenda_type": "<cost_cutter or aggressive_treater>", '
    '"agenda_evidence": "<evidence>", "agenda_counter": "<counter>"}'
)

FALLBACK_ACTION = {
    "action_type": "share_information",
    "content": "I need to share my clinical findings. Based on my private information, I believe we must reach consensus on the best treatment for this patient.",
    "reasoning": "Sharing private information to progress the negotiation toward a correct joint decision.",
}
FLAG_FALLBACK_FIELDS = {
    "bias_location": "not specified", "bias_direction": "not specified", "bias_correction": "not specified",
    "agenda_type": "cost_cutter", "agenda_evidence": "not specified", "agenda_counter": "not specified",
}

# ── Local reward function ─────────────────────────────────────────────────────
def compute_rewards(prompts, completions, task_ids=None, **kwargs):
    rewards = []
    for completion in completions:
        score, parsed = 0.0, None
        try:
            parsed = json.loads(completion)
        except Exception:
            pass
        if parsed is None:
            try:
                m = re.search(r"\{[^{}]*\}", completion, re.DOTALL)
                if m: parsed = json.loads(m.group())
            except Exception:
                pass
        if isinstance(parsed, dict):
            score += 0.3
            atype   = parsed.get("action_type", "")
            content = parsed.get("content", "")
            reason  = parsed.get("reasoning", "")
            if atype in VALID_ACTIONS:        score += 0.2
            if len(str(content))  > 50:       score += 0.2
            if len(str(reason))   > 30:       score += 0.2
            if any(k in str(content).lower() for k in MEDICAL_KEYWORDS): score += 0.1
            if atype in ADVERSARIAL_ACTIONS:  score = min(1.0, score + 0.15)
        rewards.append(round(min(1.0, max(0.0, score)), 4))
    return rewards

# ── Rollout ───────────────────────────────────────────────────────────────────
def _parse_action(text):
    try:
        p = json.loads(text)
        if isinstance(p, dict): return p
    except Exception: pass
    try:
        m = re.search(r"\{[^{}]*\}", text, re.DOTALL)
        if m:
            p = json.loads(m.group())
            if isinstance(p, dict): return p
    except Exception: pass
    return None

def _rollout(model, tokenizer, task_id):
    all_prompts, all_completions, history = [], [], []
    session_id = None
    try:
        resp = requests.post(f"{SPACE_URL}/reset", json={"task_id": task_id}, timeout=30)
        resp.raise_for_status()
        state = resp.json(); session_id = state.get("session_id")
    except Exception as e:
        print(f"  Reset failed {task_id}: {e}"); return [], [], 0.05
    obs_a = state.get("obs_agent_a", {}); obs_b = state.get("obs_agent_b", {})

    for turn in range(MAX_TURNS):
        agent_id = "agent_a" if turn % 2 == 0 else "agent_b"
        obs      = obs_a if agent_id == "agent_a" else obs_b
        private_info = json.dumps(obs.get("private_information", {}), indent=2)[:600]
        available    = obs.get("available_actions", VALID_ACTIONS)
        history_str  = "".join(f"  [{m['agent_id']}] {m['action_type']}: {m['content'][:200]}\n" for m in history[-6:]) or "  (No messages yet.)"
        prompt = (SYSTEM_PROMPT + "\n\n"
            f"Task: {obs.get('task_description','Medical negotiation.')}\n"
            f"Phase: {obs.get('current_phase','triage')} (turn {obs.get('phase_turn',0)})\n"
            f"Private info:\n{private_info}\n\nConversation:\n{history_str}\n"
            f"Available: {available}\nYour turn as {agent_id}. Respond JSON only.")
        try:
            inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1800).to(model.device)
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=256, do_sample=True,
                                     temperature=0.7, top_p=0.9, pad_token_id=tokenizer.pad_token_id)
            completion = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        except Exception as e:
            print(f"  Gen failed t{turn}: {e}"); completion = json.dumps(FALLBACK_ACTION)
        all_prompts.append(prompt); all_completions.append(completion)
        parsed = _parse_action(completion)
        action = parsed if (parsed and parsed.get("action_type") in VALID_ACTIONS) else dict(FALLBACK_ACTION)
        if action["action_type"] not in available:
            action["action_type"] = "share_information" if "share_information" in available else (available[0] if available else "share_information")
        payload = {"session_id": session_id, "action": {
            "agent_id": agent_id, "action_type": action["action_type"],
            "content": str(action.get("content", FALLBACK_ACTION["content"]))[:1000],
            "reasoning": str(action.get("reasoning", FALLBACK_ACTION["reasoning"]))[:500],
        }}
        if action["action_type"] == "flag_bias":
            for f in ("bias_location","bias_direction","bias_correction"):
                payload["action"][f] = str(action.get(f, FLAG_FALLBACK_FIELDS[f]))[:300]
        if action["action_type"] == "flag_agenda":
            for f in ("agenda_type","agenda_evidence","agenda_counter"):
                payload["action"][f] = str(action.get(f, FLAG_FALLBACK_FIELDS[f]))[:300]
        try:
            sr = requests.post(f"{SPACE_URL}/step", json=payload, timeout=30)
            if sr.status_code in (503, 422): continue
            sr.raise_for_status(); step = sr.json()
        except Exception as e:
            print(f"  Step failed t{turn}: {e}"); continue
        obs_a = step.get("obs_agent_a", obs_a); obs_b = step.get("obs_agent_b", obs_b)
        history.append({"agent_id": agent_id, "action_type": action["action_type"], "content": action.get("content","")})
        if step.get("done"):
            ep = step.get("episode_result") or {}
            if ep.get("total_reward"): return all_prompts, all_completions, float(ep["total_reward"])
            try:
                s = requests.get(f"{SPACE_URL}/state", params={"session_id":session_id}, timeout=30).json()
                return all_prompts, all_completions, float(s.get("cumulative_reward", 0.05))
            except Exception: return all_prompts, all_completions, 0.05
    try:
        s = requests.get(f"{SPACE_URL}/state", params={"session_id":session_id}, timeout=30).json()
        return all_prompts, all_completions, float(s.get("cumulative_reward", 0.05))
    except Exception: return all_prompts, all_completions, 0.05

# ── Collect episodes ──────────────────────────────────────────────────────────
rewards_log, all_prompts, all_taskids = {t: [] for t in TRAIN_TASKS}, [], []
model.eval()
print("=" * 55)
print("PHASE 1 — Collecting rollout episodes")
print("=" * 55)
for task_id in TRAIN_TASKS:
    print(f"\n  Task: {task_id}")
    for ep in range(EPISODES_PER_TASK):
        prompts, completions, reward = _rollout(model, tokenizer, task_id)
        rewards_log[task_id].append(reward)
        all_prompts.extend(prompts)
        all_taskids.extend([task_id] * len(prompts))
        print(f"    ep {ep+1:02d}/{EPISODES_PER_TASK} | reward={reward:.4f} | steps={len(prompts)}")

print(f"\nTotal prompts: {len(all_prompts)}")
if not all_prompts:
    print("WARNING: No prompts — using fallback")
    all_prompts = ["You are a doctor. Task: triage STEMI. Respond JSON: {action_type, content, reasoning}"] * 16
    all_taskids = ["single-round-consensus"] * 16

dataset = Dataset.from_dict({"prompt": all_prompts, "task_ids": all_taskids})
print(f"Dataset: {len(dataset)} samples")

# ── GRPO update ───────────────────────────────────────────────────────────────
config = GRPOConfig(
    output_dir                  = "./grpo_output",
    num_train_epochs            = N_EPOCHS,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    learning_rate               = 2e-5,
    warmup_ratio                = 0.05,
    max_completion_length       = 256,
    num_generations             = GROUP_SIZE,
    logging_steps               = 1,
    save_steps                  = 50,
    report_to                   = "none",
)

model.train()
trainer = GRPOTrainer(
    model            = model,
    processing_class = tokenizer,
    reward_funcs     = [compute_rewards],
    args             = config,
    train_dataset    = dataset,
)

print("\n" + "=" * 55)
print("PHASE 2 — GRPO Training")
print("=" * 55)
trainer.train()
print("\nTraining complete!")
for task_id, scores in rewards_log.items():
    if scores:
        print(f"  {task_id:<35} avg={sum(scores)/len(scores):.4f}  best={max(scores):.4f}")

In [ ]:
# Cell 8 — Plot reward curve
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

TASK_COLORS = {
    "single-round-consensus" : "#6366f1",
    "adversarial-information": "#ef4444",
    "opioid-overdose"        : "#22c55e",
}

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor("#0d1117"); ax.set_facecolor("#0d1117")
has_data = False

for task_id, scores in rewards_log.items():
    if not scores: continue
    has_data = True
    color    = TASK_COLORS.get(task_id, "#94a3b8")
    episodes = list(range(1, len(scores) + 1))
    ax.plot(episodes, scores, color=color, alpha=0.25, linewidth=1.2)
    ax.scatter(episodes, scores, color=color, s=22, alpha=0.5, zorder=3)
    if len(scores) >= 3:
        smoothed = np.convolve(scores, np.ones(3)/3, mode="valid")
        ax.plot(range(2, len(scores)+1), smoothed, color=color, linewidth=2.5,
                linestyle="--", label=f"{task_id} (smoothed)", zorder=4)
    else:
        ax.plot(episodes, scores, color=color, linewidth=2.5, linestyle="--", label=task_id, zorder=4)

ax.set_xlabel("Episode", color="white", fontsize=13)
ax.set_ylabel("Reward Score", color="white", fontsize=13)
ax.set_title("GRPO Training — Social Agent Negotiation", color="white", fontsize=14, fontweight="bold")
ax.set_ylim(0, 1.05); ax.tick_params(colors="white")
ax.spines[["top","right"]].set_visible(False)
ax.spines[["bottom","left"]].set_color("#333")
ax.yaxis.grid(True, color="#1e1e2e", linewidth=0.8); ax.set_axisbelow(True)
if has_data:
    ax.legend(facecolor="#161b22", edgecolor="#333", labelcolor="white", fontsize=11, loc="lower right")
plt.tight_layout()
plt.savefig("reward_curve_grpo.png", dpi=150, bbox_inches="tight", facecolor="#0d1117")
plt.show()
print("Saved: reward_curve_grpo.png")

In [ ]:
# Cell 9 — Save and push to HuggingFace Hub
import os, builtins, shutil
from huggingface_hub import HfApi

HF_TOKEN = getattr(builtins, "hf_token", os.environ.get("HF_TOKEN", ""))
HF_REPO  = "Bharath-1608/negotiation-agent-grpo"
SAVE_DIR = "negotiation_lora_final"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

if os.path.exists("reward_curve_grpo.png"):
    shutil.copy("reward_curve_grpo.png", os.path.join(SAVE_DIR, "reward_curve_grpo.png"))

model_card = (
    "---\nbase_model: unsloth/Qwen2.5-0.5B-Instruct\n"
    "tags:\n  - negotiation\n  - medical\n  - grpo\n  - multi-agent\n  - openenv\n---\n\n"
    "# Social Agent Negotiation — GRPO LoRA (Qwen2.5-0.5B)\n\n"
    "Fine-tuned via GRPO on the social-agent-negotiation-v1 OpenEnv environment.\n"
    "Base model: unsloth/Qwen2.5-0.5B-Instruct (4-bit LoRA, r=16, alpha=32)\n"
)
with open(os.path.join(SAVE_DIR, "README.md"), "w") as f:
    f.write(model_card)

print(f"Saved to ./{SAVE_DIR}")

if not HF_TOKEN:
    print("HF_TOKEN not set — skipping push.")
else:
    api = HfApi()
    api.create_repo(HF_REPO, token=HF_TOKEN, repo_type="model", exist_ok=True)
    api.upload_folder(folder_path=SAVE_DIR, repo_id=HF_REPO, repo_type="model", token=HF_TOKEN)
    print(f"Pushed to https://huggingface.co/{HF_REPO}")